<div align="center">
<h1 style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 30px; border-radius: 20px; margin-bottom: 20px;">🔐 KRİPTOQRAFİYA KURSU</h1>
<h2 style="color: #2c3e50; font-size: 2.5em; margin: 20px 0;">Məşğələ 23</h2>
<h2 style="color: #e74c3c; font-size: 2em; margin: 20px 0;">Rəqəmsal İmza Sxemləri</h2>
<h3 style="color: #7f8c8d; font-size: 1.5em; margin: 20px 0;">RSA, DSA, ECDSA, Ed25519 və daha çoxu</h3>
<br>
<div style="background-color: #3498db; color: white; padding: 15px; border-radius: 50px; display: inline-block; font-size: 1.2em;">⏱️ Məşğələ vaxtı: 2 saat</div>
<br><br>
<p><em>Hazırlanma tarixi: 2024</em></p>
</div>

## 📑 Mündəricat

1. [Məşğələnin məqsədləri](#məşğələnin-məqsədləri)
2. [İcra mühiti](#icra-mühiti)
3. [Rəqəmsal imza və MAC](#rəqəmsal-imza-və-mac)
4. [Mesajın kodlanması və heşlənməsi](#mesajın-kodlanması-və-heşlənməsi)
5. [RSA imzaları](#rsa-imzaları)
6. [DSA və per-message secret](#dsa-və-per-message-secret)
7. [ECDSA və protokol formatları](#ecdsa-və-protokol-formatları)
8. [Ed25519](#ed25519)
9. [Qabaqcıl imza sxemləri](#qabaqcıl-imza-sxemləri)
10. [Praktik seçim sərhədləri](#praktik-seçim-sərhədləri)
11. [İnteqrasiya edilmiş özünü-yoxlama](#inteqrasiya-edilmiş-özünü-yoxlama)
12. [Ev tapşırığı](#ev-tapşırığı)
13. [Yekun və mənbələr](#yekun-və-mənbələr)


## 🎯 Məşğələnin məqsədləri

Bu məşğələni bitirdikdən sonra tələbə:

- rəqəmsal imzanın kriptoqrafik yoxlamasını açar sahibinin kimliyi, vaxt, etimad və hüquqi nəticədən ayırır;
- RSA-PSS, legacy DSA verification, ECDSA və Ed25519-un əsas təhlükəsizlik sərhədlərini izah edir;
- dəyişdirilmiş mesaj və imzanın rəddini avtomatik testlərlə yoxlayır;
- ECDSA-nın DER və sabit enli `r || s` formatlarını, həmçinin Bitcoin ECDSA ilə BIP340 Schnorr-u fərqləndirir;
- per-message secret təkrarından DSA/ECDSA private key-in necə bərpa olunduğunu hesablayır;
- multisig, threshold signature, ring signature, blind signature və hash-based signature anlayışlarını qarışdırmır;
- sxemi yalnız protokol profili, uyğunluq, implementation assurance və key-management tələbləri məlum olduqda seçir.


## 📚 İcra mühiti

| Asılılıq | Tələb | İstifadə sahəsi |
|---|---|---|
| Python | 3.10+ | standart kitabxana, ölçmə və assertions |
| `cryptography` | cari kernel-də əvvəlcədən quraşdırılmış | RSA-PSS, DSA, ECDSA, Ed25519 |

> **Mühit müqaviləsi:** notebook şəbəkədən paket quraşdırmır, cari qovluğu dəyişdirmir, fayl yaratmır və `input()` gözləmir. Məcburi asılılıq yoxdursa icra ilk mərhələdə aydın xəta ilə dayanır; təhlükəsizlik mövzusu availability flag ilə səssiz buraxılmır.


In [ ]:
# Fail-fast dependency contract: notebook daxilində paket quraşdırılmır
import sys
from importlib.util import find_spec

assert sys.version_info >= (3, 10), "Python 3.10+ tələb olunur"
if find_spec("cryptography") is None:
    raise RuntimeError("Tələb olunan paket yoxdur: cryptography")

print(f"✅ Python {sys.version_info.major}.{sys.version_info.minor}; cryptography tapıldı")


## 🔧 Hazırlıq (15 dəq)

### 3.1 Python mühitinin yoxlanılması

Aşağıdakı hüceyrə notebook-da istifadə olunan bütün public API-ləri bir yerdə import edir. Import xətası optional nəticə deyil və icranı dayandırır.


In [ ]:
import hashlib
import statistics
import time

import cryptography
from cryptography.exceptions import InvalidSignature
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import dsa, ec, ed25519, padding, rsa
from cryptography.hazmat.primitives.asymmetric.utils import (
    Prehashed,
    decode_dss_signature,
    encode_dss_signature,
)

print(f"✅ cryptography {cryptography.__version__}: tələb olunan API-lər import edildi")


### 3.2 Təkrarolunan icra qaydası

Notebook gizli açarları, private komponentləri və imza üçün yaradılan secret dəyərləri ekrana çıxarmır. Nəticələr yalnız public ölçülər, yoxlama statusu və lokal ölçmənin medianı ilə təqdim edilir.


In [ ]:
execution_contract = {
    "installs_packages": False,
    "changes_working_directory": False,
    "creates_files": False,
    "waits_for_input": False,
    "prints_private_material": False,
}
assert not any(execution_contract.values())
print("✅ İcra müqaviləsi yoxlanıldı")


## 📖 Xatırlatma: Rəqəmsal imza vs MAK (5 dəq)

<a id="rəqəmsal-imza-və-mac"></a>
## 📖 Rəqəmsal imza və MAC

| Xüsusiyyət | MAC | Rəqəmsal imza |
|---|---|---|
| Açar modeli | ortaq secret key | private signing key + public verification key |
| Kim yarada bilər | secret-i bilən hər tərəf | private key-ə nəzarət edən tərəf |
| Kim yoxlaya bilər | secret-i bilən tərəf | public key və parametr/profilə malik verifier |
| Üçüncü tərəfə sübut | adətən zəifdir: verifier də MAC yarada bilir | əlavə assurance olduqda mümkün ola bilər |
| Əsas kriptoqrafik nəticə | bütövlük və ortaq-açar autentifikasiyası | imzanın verilmiş public key, mesaj və profil üzrə yoxlanması |

Uğurlu `verify` əməliyyatı öz-özünə şəxsin kimliyini, imzalama vaxtını, razılığı və ya hüquqi “inkar edilməzliyi” sübut etmir. Bunun üçün public key-in sahibinə etibarlı bağlanması, private-key possession, alqoritm/format identifikasiyası, sertifikat və revocation siyasəti, lazım olduqda etibarlı vaxt möhürü və tətbiq qaydaları ayrıca tələb olunur.


## 🔄 Heş-then-Sign paradiqması (5 dəq)

<a id="mesajın-kodlanması-və-heşlənməsi"></a>
## 🔄 Mesajın kodlanması və heşlənməsi

Bir çox imza profili əvvəlcə mesajı deterministik bayt sətrinə kodlayır və digest hesablayır; RSA-PSS və ECDSA API-ləri hash funksiyasını qəbul edir və ya `Prehashed` rejimini açıq göstərir. Lakin bu, bütün API-lərin “xaricdə heşlə, sonra həmin baytları adi mesaj kimi yenidən imzala” demək deyil:

- protokol imzalanan dəqiq baytları, domain separation-u və alqoritm identifikatorunu müəyyən etməlidir;
- `Prehashed` yalnız digest və hash alqoritmi həqiqətən profildə sabitləndikdə işlədilməlidir;
- Ed25519 mesajı öz RFC 8032 signing proseduruna daxil edir; Ed25519, Ed25519ctx və Ed25519ph ayrı variantlardır;
- signature verification yalnız eyni encoding, variant, public key və alqoritm parametrləri ilə mənalıdır.

Beləliklə, təhlükəsiz qayda “həmişə əl ilə hash-then-sign” deyil, **standartlaşdırılmış signing API və protokol profilinin imzaladığı dəqiq baytları dəyişməməkdir**.


## 🔐 RSA imza sxemi (20 dəq)

<a id="rsa-imzaları"></a>
## 🔐 RSA imzaları

### 5.1 Textbook RSA: yalnız cəbri demonstrasiya

Aşağıdakı kiçik nümunə standart imza sxemi deyil. O, `m ↦ m^d mod n` əməliyyatının düzgünlüyünü və encoding/padding olmadan existential forgery mümkünlüyünü göstərir. Private exponent ekrana çıxarılmır; production RSA ən azı 2048-bit modul və standartlaşdırılmış RSASSA encoding tələb edir.


In [ ]:
def rsa_basic_signature_demo():
    """Kiçik textbook RSA-nın cəbrini və existential forgery-ni yoxlayır."""
    p, q, e = 61, 53, 17
    n = p * q
    phi = (p - 1) * (q - 1)
    d = pow(e, -1, phi)

    message = 42
    signature = pow(message, d, n)
    recovered = pow(signature, e, n)
    assert recovered == message

    # Hücumçu əvvəl signature seçir, sonra ona uyğun message-i hesablayır.
    # Bu, əvvəlcədən seçilmiş konkret mesaja imza deyil, existential forgery-dir.
    forged_signature = 123
    forged_message = pow(forged_signature, e, n)
    assert pow(forged_signature, e, n) == forged_message

    result = {
        "public_modulus": n,
        "public_exponent": e,
        "round_trip": True,
        "existential_forgery_pair": True,
    }
    print("✅ Textbook RSA: round-trip və existential forgery cütü yoxlanıldı; private d göstərilmir")
    return result


rsa_basic_result = rsa_basic_signature_demo()


### 5.2 RSA-PSS (Probabilistic Signature Scheme)

In [ ]:
def rsa_pss_parameters():
    return padding.PSS(
        mgf=padding.MGF1(hashes.SHA256()),
        salt_length=padding.PSS.DIGEST_LENGTH,
    )


def rsa_pss_demo():
    private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
    public_key = private_key.public_key()
    message = b"AS23 RSA-PSS test message"
    signature = private_key.sign(message, rsa_pss_parameters(), hashes.SHA256())
    second_signature = private_key.sign(message, rsa_pss_parameters(), hashes.SHA256())

    public_key.verify(signature, message, rsa_pss_parameters(), hashes.SHA256())

    changed_message_rejected = False
    try:
        public_key.verify(signature, message + b"!", rsa_pss_parameters(), hashes.SHA256())
    except InvalidSignature:
        changed_message_rejected = True

    tampered = bytearray(signature)
    tampered[-1] ^= 1
    changed_signature_rejected = False
    try:
        public_key.verify(bytes(tampered), message, rsa_pss_parameters(), hashes.SHA256())
    except InvalidSignature:
        changed_signature_rejected = True

    assert signature != second_signature
    assert changed_message_rejected and changed_signature_rejected
    result = {
        "modulus_bits": private_key.key_size,
        "signature_length": len(signature),
        "probabilistic": True,
        "message_tamper_rejected": True,
        "signature_tamper_rejected": True,
    }
    print("✅ RSA-PSS: positive verify, probabilistik nəticə və iki negative test keçdi")
    return private_key, public_key, message, signature, result


(
    rsa_pss_private_key,
    rsa_pss_public_key,
    rsa_pss_message,
    rsa_pss_signature,
    rsa_pss_result,
) = rsa_pss_demo()


### 5.3 RSASSA-PKCS1-v1_5 və RSASSA-PSS

Hər ikisi RFC 8017-də RSA **signature** sxemidir; onları eyni adlı RSA encryption sxemləri ilə qarışdırmaq olmaz.

| Sərhəd | RSASSA-PKCS1-v1_5 | RSASSA-PSS |
|---|---|---|
| Encoding | deterministik `DigestInfo` əsaslı | salt ilə probabilistik |
| Profil uyğunluğu | bir çox legacy protokolda tələb oluna bilər | yeni profillərdə tez-tez seçilir |
| Təhlükəsizlik arqumenti | RFC 8017 ayrıca müzakirə edir; “heç bir sübut yoxdur” kimi mütləq hökm düzgün deyil | RFC 8017 random-oracle modelində daha güclü əsaslandırma verir |
| Seçim | mövcud profil tələb edirsə, dəqiq DER/hash qaydaları ilə | profil imkan verirsə, hash/MGF/salt parametrləri tam sabitlənərək |

Alqoritm seçimi qarşı tərəfin profili və compliance tələbindən ayrılıqda edilmir. Eyni RSA key pair-i encryption və signature kimi müxtəlif məqsədlər üçün təkrar istifadə etmək də düzgün key-separation deyil.


### ✍️ Çalışma 1: RSA imzası (1 bal)

1. RSA-PSS/SHA-256 ilə mesajı imzalayın və public key ilə yoxlayın.
2. Eyni mesajı ikinci dəfə imzalayıb salt səbəbindən nəticələrin fərqli olduğunu yoxlayın.
3. Dəyişdirilmiş mesajı və bir biti dəyişdirilmiş imzanı ayrı-ayrılıqda rədd edin.
4. Textbook RSA demonstrasiyasındakı existential forgery-ni əvvəlcədən seçilmiş mesaja forgery-dən fərqləndirin.
5. RSASSA ilə RSAES adlarının niyə bir-birini əvəz etmədiyini izah edin.

**Yoxlama meyarı:** positive verify keçir; iki negative verify `InvalidSignature` ilə rədd edilir; private komponent output-da yoxdur; cavab RFC 8017 terminlərini tam adla işlədir.


## 📝 DSA (Digital Signature Algorithm) (15 dəq)

<a id="dsa-və-per-message-secret"></a>
## 📝 DSA və per-message secret

### 6.1 DSA-nın riyazi modeli və cari standart statusu

DSA, `q | (p-1)` olan sonlu meydan subgroup-unda diskret loqarifm probleminə əsaslanır. Private key `x ∈ [1,q-1]`, public key `y = g^x mod p`-dir. Mesaj digest-indən alınan `z` və hər imza üçün gizli `k ∈ [1,q-1]` ilə:

`r = (g^k mod p) mod q`, `s = k⁻¹(z + xr) mod q`.

Verifier `0 < r,s < q` aralığını yoxlayır, sonra `w=s⁻¹`, `u₁=zw`, `u₂=rw` və `v=((g^u₁ y^u₂ mod p) mod q)` hesablayıb `v=r` tələb edir.

**Standart sərhədi:** FIPS 186-5 DSA-nı yeni signature generation üçün artıq təsdiqləmir; yalnız standartın tətbiq tarixindən əvvəl yaradılmış imzaların verification-ına icazə verir. Aşağıdakı library nümunəsi riyazi/legacy interoperability tədrisidir və yeni sistem üçün compliance tövsiyəsi deyil.


In [ ]:
def dsa_demo():
    """Legacy DSA API-sində formatı, positive və negative verify-ni yoxlayır."""
    private_key = dsa.generate_private_key(key_size=2048)
    public_key = private_key.public_key()
    message = b"AS23 legacy DSA verification example"
    signature = private_key.sign(message, hashes.SHA256())
    public_key.verify(signature, message, hashes.SHA256())

    r, s = decode_dss_signature(signature)
    assert encode_dss_signature(r, s) == signature

    changed_message_rejected = False
    try:
        public_key.verify(signature, message + b"!", hashes.SHA256())
    except InvalidSignature:
        changed_message_rejected = True

    tampered = bytearray(signature)
    tampered[-1] ^= 1
    changed_signature_rejected = False
    try:
        public_key.verify(bytes(tampered), message, hashes.SHA256())
    except InvalidSignature:
        changed_signature_rejected = True

    assert changed_message_rejected and changed_signature_rejected
    result = {
        "key_bits": private_key.key_size,
        "der_round_trip": True,
        "message_tamper_rejected": True,
        "signature_tamper_rejected": True,
        "current_fips_generation_profile": False,
    }
    print("✅ Legacy DSA: DER round-trip, positive verify və iki negative test keçdi")
    return private_key, public_key, message, signature, result


dsa_private_key, dsa_public_key, dsa_message, dsa_signature, dsa_result = dsa_demo()


### 6.2 Eyni per-message secret-in təkrarı

DSA və ECDSA-da eyni `k` iki fərqli digest üçün işlədilərsə və eyni `r` alınarsa:

`k = (z₁-z₂)(s₁-s₂)⁻¹ mod q`,

`x = (s₁k-z₁)r⁻¹ mod q`.

Bu bərpa yalnız lazımi modulyar inverslər mövcud olduqda və iki imza həqiqətən eyni secret ilə yaradıldıqda tətbiq edilir. Deterministik per-message secret generation random-generator nasazlığı riskini azalda bilər, amma domain separation, fault/side-channel müdafiəsi və düzgün implementasiyanı avtomatik təmin etmir. ECDSA üçün FIPS 186-5 həm random, həm deterministik generation metodlarını müəyyən edir.


In [ ]:
def toy_dsa_sign(p, q, g, private_x, secret_k, digest_z):
    r = pow(g, secret_k, p) % q
    s = (pow(secret_k, -1, q) * (digest_z + private_x * r)) % q
    assert 0 < r < q and 0 < s < q
    return r, s


def toy_dsa_verify(p, q, g, public_y, digest_z, signature):
    r, s = signature
    if not (0 < r < q and 0 < s < q):
        return False
    w = pow(s, -1, q)
    u1 = digest_z * w % q
    u2 = r * w % q
    return (pow(g, u1, p) * pow(public_y, u2, p) % p) % q == r


def k_reuse_problem_demo():
    p, q, g = 23, 11, 2
    private_x, reused_k = 6, 7
    z1, z2 = 9, 3
    public_y = pow(g, private_x, p)

    signature1 = toy_dsa_sign(p, q, g, private_x, reused_k, z1)
    signature2 = toy_dsa_sign(p, q, g, private_x, reused_k, z2)
    r1, s1 = signature1
    r2, s2 = signature2
    assert r1 == r2
    assert toy_dsa_verify(p, q, g, public_y, z1, signature1)
    assert toy_dsa_verify(p, q, g, public_y, z2, signature2)

    recovered_k = ((z1 - z2) * pow((s1 - s2) % q, -1, q)) % q
    recovered_x = ((s1 * recovered_k - z1) * pow(r1, -1, q)) % q
    assert recovered_k == reused_k and recovered_x == private_x

    result = {
        "two_signatures_valid": True,
        "same_r": True,
        "recovered_k": recovered_k,
        "recovered_private_x": recovered_x,
    }
    print("✅ Toy DSA: iki düzgün imzadan reused k və private x tam bərpa edildi")
    return result


k_reuse_result = k_reuse_problem_demo()


### ✍️ Çalışma 2: DSA və nonce reuse (1 bal)

1. Legacy DSA nümunəsində positive verify və iki negative verify-ni təkrar edin.
2. FIPS 186-5 üzrə yeni DSA signature generation statusunu bir cümlə ilə düzgün yazın.
3. Toy parametrlərdə iki imzanı ayrıca verify edin, sonra eyni `r` şərtini və hər iki inversin mövcudluğunu yoxlayıb `k` və `x`-i bərpa edin.
4. Deterministik generation-un hansı riski azaltdığını və hansı implementation risklərini həll etmədiyini izah edin.

**Yoxlama meyarı:** hər iki ilkin imza düzgündür; bərpa olunan `k` və `x` başlanğıc qiymətlərinə bərabərdir; compliance hökmü FIPS 186-5 ilə uyğun gəlir.


## ⚡ ECDSA (Elliptic Curve Digital Signature Algorithm) (15 dəq)

<a id="ecdsa-və-protokol-formatları"></a>
## ⚡ ECDSA və protokol formatları

### 7.1 ECDSA-nın iş prinsipi

Elliptik əyri subgroup-u, baza nöqtəsi `G` və onun sadə sırası `n` üçün private key `d`, public key `Q=[d]G`-dir. Hər imzada qorunan per-message secret `k` seçilir, `R=[k]G`, `r=x_R mod n`, `s=k⁻¹(z+rd) mod n` hesablanır; `r=0` və ya `s=0` olarsa yeni `k` tələb olunur. `z` hash output-unun `n`-ə uyğun çevrilməsidir.

FIPS 186-5 random və deterministik ECDSA per-message secret generation-u ayrıca müəyyən edir. Hansı variantın istifadə edilməsi API və protokol profilindən asılıdır; tətbiq kodu özbaşına `k` yaratmamalıdır.


In [ ]:
def ecdsa_demo():
    private_key = ec.generate_private_key(ec.SECP256R1())
    public_key = private_key.public_key()
    message = b"AS23 ECDSA P-256 test message"
    signature_der = private_key.sign(message, ec.ECDSA(hashes.SHA256()))
    public_key.verify(signature_der, message, ec.ECDSA(hashes.SHA256()))

    r, s = decode_dss_signature(signature_der)
    width = (private_key.curve.key_size + 7) // 8
    signature_raw = r.to_bytes(width, "big") + s.to_bytes(width, "big")
    assert len(signature_raw) == 64
    assert encode_dss_signature(r, s) == signature_der

    changed_message_rejected = False
    try:
        public_key.verify(signature_der, message + b"!", ec.ECDSA(hashes.SHA256()))
    except InvalidSignature:
        changed_message_rejected = True

    tampered = bytearray(signature_der)
    tampered[-1] ^= 1
    changed_signature_rejected = False
    try:
        public_key.verify(bytes(tampered), message, ec.ECDSA(hashes.SHA256()))
    except InvalidSignature:
        changed_signature_rejected = True

    assert changed_message_rejected and changed_signature_rejected
    result = {
        "curve": private_key.curve.name,
        "der_length": len(signature_der),
        "raw_length": len(signature_raw),
        "der_round_trip": True,
        "message_tamper_rejected": True,
        "signature_tamper_rejected": True,
    }
    print(f"✅ ECDSA P-256: DER={len(signature_der)} bayt, raw=64 bayt; negative tests keçdi")
    return private_key, public_key, message, signature_der, result


(
    ecdsa_private_key,
    ecdsa_public_key,
    ecdsa_message,
    ecdsa_signature,
    ecdsa_result,
) = ecdsa_demo()


### 7.2 secp256k1, Bitcoin ECDSA və BIP340 Schnorr

`cryptography` ilə secp256k1 üzərində ümumi ECDSA round-trip göstərmək Bitcoin transaction implementasiyası demək deyil. Bitcoin imzalanacaq transaction digest-i ayrıca consensus qaydaları ilə qurur.

- legacy və SegWit-v0 Bitcoin ECDSA imzasında `(r,s)` strict DER encoding-dədir və ona script kontekstində bir sighash-type baytı əlavə olunur; DER hissəsinin uzunluğu dəyişə bilər (BIP66);
- Taproot BIP340 Schnorr signature özü sabit 64 baytdır; BIP341 bəzi hallarda əlavə sighash baytına icazə verir;
- “Bitcoin imzası həmişə 64-bayt `r||s` ECDSA-dır” hökmü bu iki ayrı sxem və wire formatını qarışdırır.


In [ ]:
def secp256k1_ecdsa_format_demo():
    """Ümumi secp256k1 ECDSA API-si; Bitcoin transaction signer deyil."""
    private_key = ec.generate_private_key(ec.SECP256K1())
    public_key = private_key.public_key()
    digest = hashlib.sha256(b"AS23 protocol-defined payload example").digest()
    signature_der = private_key.sign(
        digest,
        ec.ECDSA(Prehashed(hashes.SHA256())),
    )
    public_key.verify(
        signature_der,
        digest,
        ec.ECDSA(Prehashed(hashes.SHA256())),
    )

    r, s = decode_dss_signature(signature_der)
    signature_raw = r.to_bytes(32, "big") + s.to_bytes(32, "big")
    compressed_public = public_key.public_bytes(
        encoding=serialization.Encoding.X962,
        format=serialization.PublicFormat.CompressedPoint,
    )
    assert len(signature_raw) == 64
    assert len(compressed_public) == 33
    assert encode_dss_signature(r, s) == signature_der

    result = {
        "curve": private_key.curve.name,
        "compressed_public_length": len(compressed_public),
        "ecdsa_der_length": len(signature_der),
        "mathematical_raw_length": len(signature_raw),
        "is_bitcoin_transaction_implementation": False,
        "bip340_schnorr_signature_length": 64,
    }
    print(
        f"✅ secp256k1 generic ECDSA: DER={len(signature_der)} bayt; "
        "BIP66 ECDSA və BIP340 Schnorr formatları mətnlə ayrıldı"
    )
    return result


secp256k1_format_result = secp256k1_ecdsa_format_demo()


### ✍️ Çalışma 3: ECDSA və formatlar (1 bal)

1. P-256 ECDSA signature yaradın, DER `(r,s)`-i decode/re-encode edin və verify edin.
2. 32-baytlıq `r` və `s` hissələrindən 64-bayt mathematical `r||s` yaradın; bunun protokol formatı olmadığını qeyd edin.
3. Dəyişdirilmiş mesajı və imzanı negative tests ilə rədd edin.
4. Bitcoin ECDSA/BIP66, əlavə sighash byte və BIP340 Schnorr 64-byte formatını ayrı-ayrılıqda izah edin.
5. `Prehashed` işlədərkən digest alqoritmi və imzalanan protocol bytes niyə əvvəlcədən sabitlənməlidir?

**Yoxlama meyarı:** DER round-trip verify keçir; iki negative test rədd edilir; cavab ECDSA ilə Schnorr-u və mathematical raw ilə wire formatını qarışdırmır.


## ⚡ Ed25519 (10 dəq)

<a id="ed25519"></a>
## ⚡ Ed25519

### 8.1 RFC 8032 sərhədləri

Ed25519, `p=2^255-19` meydanı üzərində edwards25519 twisted Edwards əyrisindən istifadə edən EdDSA instansiyasıdır. RFC forması `-x² + y² = 1 + d x²y²`-dir (`a=-1`).

- private key input/seed encoding: 32 bayt; public key encoding: 32 bayt; signature: 64 bayt;
- imza nonce-sinə bənzər `r`, yalnız mesajdan deyil, private seed-dən alınan secret prefix və mesajdan hash edilir;
- eyni key/message üçün Ed25519 deterministik nəticə verir;
- Ed25519, Ed25519ctx və Ed25519ph bir-birini əvəz etmir;
- deterministik sxem random nonce nasazlığını azaldır, amma fault, key exfiltration və side-channel risklərini avtomatik aradan qaldırmır;
- konkret sürət və constant-time assurance kitabxana, versiya, backend və platforma üzrə ölçülür/audit edilir; universal “2–3 dəfə sürətli” hökmü verilmir.


In [ ]:
def ed25519_demo():
    private_key = ed25519.Ed25519PrivateKey.generate()
    public_key = private_key.public_key()
    public_bytes = public_key.public_bytes(
        encoding=serialization.Encoding.Raw,
        format=serialization.PublicFormat.Raw,
    )
    message = b"AS23 Ed25519 test message"
    signature = private_key.sign(message)
    repeated_signature = private_key.sign(message)

    assert len(public_bytes) == 32
    assert len(signature) == 64
    assert signature == repeated_signature
    public_key.verify(signature, message)

    changed_message_rejected = False
    try:
        public_key.verify(signature, message + b"!")
    except InvalidSignature:
        changed_message_rejected = True

    tampered = bytearray(signature)
    tampered[-1] ^= 1
    changed_signature_rejected = False
    try:
        public_key.verify(bytes(tampered), message)
    except InvalidSignature:
        changed_signature_rejected = True

    assert changed_message_rejected and changed_signature_rejected
    result = {
        "public_length": len(public_bytes),
        "signature_length": len(signature),
        "deterministic_for_same_key_message": True,
        "message_tamper_rejected": True,
        "signature_tamper_rejected": True,
    }
    print("✅ Ed25519: 32-byte public key, 64-byte signature, deterministic və iki negative test")
    return private_key, public_key, message, signature, result


(
    ed25519_private_key,
    ed25519_public_key,
    ed25519_message,
    ed25519_signature,
    ed25519_result,
) = ed25519_demo()


### 8.2 Ed25519 və ECDSA müqayisəsinin düzgün sərhədi

| Mövzu | ECDSA | Ed25519 |
|---|---|---|
| Əyri/model | profilə görə Weierstrass əyrisi | RFC 8032 edwards25519 |
| Per-message secret | FIPS 186-5 random və deterministik metodları müəyyən edir | secret prefix + message ilə deterministik `r` |
| Public encoding | əyri/protokola görə, məsələn compressed P-256 33 bayt | 32 bayt |
| Signature encoding | DER və ya profilin sabit enli forması | 64 bayt |
| Variant/profil | curve, hash, encoding və low-S kimi qaydalar ayrıca sabitlənir | Ed25519/ctx/ph seçimi ayrıca sabitlənir |
| Side-channel və sürət | konkret implementasiya üzrə yoxlanır | konkret implementasiya üzrə yoxlanır |

Bu cədvəl bir sxemin hər protokolda digərindən “daha təhlükəsiz” və ya “həmişə daha sürətli” olduğunu demir. Uyğunluq, audit edilmiş implementation, key lifecycle və protokolun normativ algorithm suite-i seçimə daxildir.


### ✍️ Çalışma 4: Ed25519 (1 bal)

1. Ed25519 key pair yaradın, eyni mesajı iki dəfə imzalayın və deterministik bərabərliyi yoxlayın.
2. Public key və signature encoding uzunluqlarını assertions ilə yoxlayın.
3. Dəyişdirilmiş mesajı və imzanı ayrıca rədd edin; yalnız `InvalidSignature` gözləyin.
4. RFC 8032-də `r`-in hansı secret və public input-lardan törədildiyini izah edin.
5. Ed25519-un OpenSSH kimi profil daxilində istifadəsini sxemin hər protokolda avtomatik uyğun olması ilə qarışdırmayın; X25519/WireGuard key agreement-i Ed25519 signature nümunəsi kimi göstərməyin.

**Yoxlama meyarı:** 32-bayt public key, 64-bayt signature, eyni key/message üçün eyni signature və iki uğurlu negative test.


## 🧠 Qabaqcıl imza sxemləri (10 dəq)

### 9.1 Kor imza (Blind Signature)

Blind-signature protokolunda requester mesajı korlayır, signer korlanmış təqdimatı imzalayır, requester isə nəticəni açaraq adi verifier üçün yoxlanılan imza əldə edir. Məxfilik və unlinkability **konkret sxemin təhlükəsizlik tərifi, düzgün blinding, issuance siyasəti və metadata müdafiəsindən** asılıdır; “signer açılmış imzanı heç vaxt tanıya bilməz” universal nəticə deyil.

Tətbiqlərə bəzi e-cash və privacy-preserving credential konstruksiyaları daxildir. Seçki və pul sistemi üçün blind signature təkbaşına eligibility, double-spend, coercion resistance və auditability təmin etmir.


### 9.2 Halqa imzası (Ring Signature)

Ring signature verifier-ə imzanın göstərilən public-key dəstindəki üzvlərdən biri tərəfindən yaradıldığını sübut etməyə çalışır, imzalayanın hansının olduğunu gizlədir. Klassik ring signature adətən əvvəlcədən qrup meneceri və koordinasiya tələb etmir; bu xüsusiyyət **group signature** anlayışı ilə müqayisə edilə bilər, threshold signature ilə yox.

Anonimlik səviyyəsi, linkability, key-image və tracing xüsusiyyətləri konkret sxemdən asılıdır. “Monero ring signatures istifadə edir” cümləsi bütün Monero privacy modelini və ya hər ring sxeminin eyni təminatları verdiyini bildirmir.


### 9.3 Hədd imzası və multisig fərqi

Threshold signature-da iştirakçılar ümumi private key-in paylarını saxlayır və ən azı `t` iştirakçı protokolda əməkdaşlıq etdikdə bir signature yaradır. Məsələn, RFC 9591 FROST iki raundlu threshold Schnorr protokoludur; uyğun ciphersuite-də nəticə adi tək-açar verifier ilə yoxlanıla bilər.

Adi multisig-də isə bir neçə müstəqil public key və bir neçə müstəqil signature protokol/script tərəfindən ayrıca qiymətləndirilə bilər. “2-of-3 wallet” yazısı onun avtomatik threshold signature olması demək deyil. Shamir secret sharing yalnız share/reconstruction primitive-idir; onu signing protocol, distributed key generation və ya proactive refresh ilə eyniləşdirmək olmaz.


### 9.4 One-time və hash-based signatures

- **Lamport OTS:** bir key pair yalnız təhlükəsizlik profilinin icazə verdiyi sayda, tipik olaraq bir dəfə işlədilir; state reuse fəlakətli ola bilər.
- **Merkle-tree yanaşmaları:** çoxlu one-time public key-i bir root altında bağlaya bilər, amma stateful variantlar signing index-in etibarlı idarəsini tələb edir.
- **SLH-DSA:** NIST FIPS 205-də standartlaşdırılmış stateless hash-based signature standardıdır və SPHINCS+ üzərində qurulub; böyük signature və performans trade-off-ları profil seçimində nəzərə alınır.

Lamport, Merkle və SLH-DSA eyni format/API deyil. “Post-kvantdır” etiketi state management, parameter set, side-channel, key storage və migration planını əvəz etmir.


<a id="praktik-seçim-sərhədləri"></a>
## 💡 Praktik seçim sərhədləri

Sxem adından əvvəl tətbiqin **normativ profili** seçilir:

1. qarşı tərəflər və mövcud protokol hansı algorithm identifier, curve/parameter set, hash, encoding və canonicalization qaydalarını tələb edir;
2. public key kimliyə necə bağlanır, revocation və key-compromise hadisəsi necə idarə olunur;
3. implementation və backend hansı versiyada audit/FIPS/constant-time assurance verir;
4. imzalanan dəqiq baytlar, domain separation, replay context və verification policy nədir;
5. key generation, backup, rotation və hardware protection üçün əməliyyat tələbləri nədir.

Mümkün nümunələr: profil icazə verirsə RSA-PSS; ekosistem tələb edirsə dəqiq parametrli ECDSA; RFC 8032 uyğunluğu tələb edilirsə Ed25519. DSA yeni FIPS signature generation üçün seçim deyil. Bitcoin legacy/SegWit ECDSA və Taproot BIP340 Schnorr ayrıca consensus profilləridir. Post-kvant migration üçün FIPS 204 ML-DSA və FIPS 205 SLH-DSA kimi standartlar qiymətləndirilir, amma mövcud protokola özbaşına əlavə edilmir.


<a id="inteqrasiya-edilmiş-özünü-yoxlama"></a>
## 🖥️ İnteqrasiya edilmiş özünü-yoxlama

Aşağıdakı hüceyrə `input()` olmadan bütün məcburi nəticələri yoxlayır və dörd local sign+verify yolunu yeddi raund ölçür. Median yalnız bu kernel, backend, platforma və mesaj üçün müşahidədir; universal performans sıralaması deyil.


In [ ]:
def benchmark_signature_round_trip(name, sign_once, verify_once, rounds=7):
    sign_samples_ns = []
    verify_samples_ns = []
    for _ in range(rounds):
        start = time.perf_counter_ns()
        signature = sign_once()
        sign_samples_ns.append(time.perf_counter_ns() - start)

        start = time.perf_counter_ns()
        verify_once(signature)
        verify_samples_ns.append(time.perf_counter_ns() - start)
    return {
        "name": name,
        "rounds": rounds,
        "median_sign_ms": statistics.median(sign_samples_ns) / 1_000_000,
        "median_verify_ms": statistics.median(verify_samples_ns) / 1_000_000,
    }


def integrated_self_test():
    assert rsa_basic_result["round_trip"] and rsa_basic_result["existential_forgery_pair"]
    assert rsa_pss_result["message_tamper_rejected"]
    assert dsa_result["der_round_trip"] and not dsa_result["current_fips_generation_profile"]
    assert k_reuse_result["recovered_k"] == 7
    assert k_reuse_result["recovered_private_x"] == 6
    assert ecdsa_result["raw_length"] == 64 and ecdsa_result["der_round_trip"]
    assert not secp256k1_format_result["is_bitcoin_transaction_implementation"]
    assert ed25519_result["public_length"] == 32
    assert ed25519_result["signature_length"] == 64

    benchmarks = [
        benchmark_signature_round_trip(
            "RSA-PSS",
            lambda: rsa_pss_private_key.sign(
                rsa_pss_message, rsa_pss_parameters(), hashes.SHA256()
            ),
            lambda sig: rsa_pss_public_key.verify(
                sig, rsa_pss_message, rsa_pss_parameters(), hashes.SHA256()
            ),
        ),
        benchmark_signature_round_trip(
            "DSA (legacy demo)",
            lambda: dsa_private_key.sign(dsa_message, hashes.SHA256()),
            lambda sig: dsa_public_key.verify(sig, dsa_message, hashes.SHA256()),
        ),
        benchmark_signature_round_trip(
            "ECDSA P-256",
            lambda: ecdsa_private_key.sign(ecdsa_message, ec.ECDSA(hashes.SHA256())),
            lambda sig: ecdsa_public_key.verify(
                sig, ecdsa_message, ec.ECDSA(hashes.SHA256())
            ),
        ),
        benchmark_signature_round_trip(
            "Ed25519",
            lambda: ed25519_private_key.sign(ed25519_message),
            lambda sig: ed25519_public_key.verify(sig, ed25519_message),
        ),
    ]
    assert all(row["rounds"] == 7 for row in benchmarks)
    for row in benchmarks:
        print(
            f"{row['name']:18} sign median={row['median_sign_ms']:.3f} ms; "
            f"verify median={row['median_verify_ms']:.3f} ms ({row['rounds']} raund)"
        )
    print("✅ AS23 self-test: bütün assertions və təkrar ölçmələr keçdi")
    return benchmarks


signature_benchmarks = integrated_self_test()


## 🏠 Ev tapşırığı (7 bal)

### 1. Profilə bağlı signature paketi (3 bal)

RSA-PSS, P-256 ECDSA və Ed25519 üçün `sign`/`verify` interfeysi hazırlayın. Hər profil algorithm identifier, hash/variant, encoding və imzalanan baytların canonicalization qaydasını sabitləməlidir.

- **1 bal:** positive və dəyişdirilmiş mesaj/imza negative tests;
- **1 bal:** DER/raw/Ed25519 format sərhədlərinin testləri və private materialın output-a düşməməsi;
- **1 bal:** README-də trust, key identity, key lifecycle və protokol compatibility məhdudiyyətləri.

Legacy DSA ayrıca verification-only adapter kimi sənədləşdirilə bilər; yeni signing profilinə daxil edilməməlidir.

### 2. Təkrarolunan benchmark (2 bal)

Ən azı 30 warm-up-dan sonra hər sxem üçün ən azı 100 sign və 100 verify nümunəsi ölçün. Median və p95, Python/library/backend/OS/CPU, mesaj ölçüsü və key/parameter set-i göstərin. Nəticəni universal “ən sürətli” hökmünə çevirməyin.

- **1 bal:** metodika və raw samples;
- **1 bal:** düzgün statistika, mühit və məhdudiyyətlər.

### 3. Mənbəli araşdırma (2 bal)

FIPS 186-5, RFC 8032, BIP66/BIP340 və RFC 9591-dən birbaşa istifadə edərək DSA statusu, Ed25519 variantları, Bitcoin signature formatları və threshold-vs-multisig fərqini müqayisə edin.

- **1 bal:** ən azı dörd primary source-a dəqiq istinad;
- **1 bal:** hər iddianın scope-u və “verification ≠ identity/legal effect” sərhədi.


## 📌 Yekun

- Signature verification verilmiş public key, mesaj və profil üzrə riyazi nəticədir; identity, vaxt və hüquqi effekt əlavə assurance tələb edir.
- Textbook RSA standart RSASSA deyil; RSA-PSS encoding/hash/MGF/salt profilini tələb edir.
- DSA riyaziyyatı nonce-reuse riskini yaxşı göstərir, lakin FIPS 186-5 yeni DSA signature generation-u təsdiqləmir.
- ECDSA DER, mathematical `r||s` və protokol wire formatı ayrı anlayışlardır; Bitcoin ECDSA ilə BIP340 Schnorr qarışdırılmır.
- Ed25519 32-bayt public key və 64-bayt signature verir, amma determinism bütün implementation risklərini həll etmir.
- Threshold signature, multisig, ring, blind və hash-based signatures ayrı təhlükəsizlik məqsədləri və əməliyyat sərhədlərinə malikdir.

Müzakirə üçün: verifier-in public key-i etibarlı identifikatora bağlaması, imzalanan baytları yenidən qurması və algorithm downgrade-i rədd etməsi hansı qatlarda həyata keçirilməlidir?


<a id="yekun-və-mənbələr"></a>
## 📚 Əlavə resurslar

Əsas və normativ mənbələr:

- RFC 8017 — RSAES/RSASSA və PKCS #1 v2.2: https://www.rfc-editor.org/rfc/rfc8017
- FIPS 186-5 — RSA, ECDSA, EdDSA və legacy DSA verification statusu: https://doi.org/10.6028/NIST.FIPS.186-5
- RFC 8032 — Ed25519/Ed448: https://www.rfc-editor.org/rfc/rfc8032
- BIP66 — Bitcoin strict DER ECDSA encoding: https://github.com/bitcoin/bips/blob/master/bip-0066.mediawiki
- BIP340 — 64-bayt Schnorr signatures over secp256k1: https://github.com/bitcoin/bips/blob/master/bip-0340.mediawiki
- BIP341 — Taproot signature və sighash qaydaları: https://github.com/bitcoin/bips/blob/master/bip-0341.mediawiki
- RFC 9591 — FROST threshold Schnorr: https://www.rfc-editor.org/rfc/rfc9591
- FIPS 205 — SLH-DSA stateless hash-based signatures: https://doi.org/10.6028/NIST.FIPS.205
- Chaum (1982), “Blind Signatures for Untraceable Payments”
- Rivest, Shamir, Tauman (2001), “How to Leak a Secret”

> Kod tədris üçündür. Real sistemdə normativ protokol profili, audit edilmiş kitabxana, key-management, identity binding, revocation və verification policy tələb olunur.

---

✅ **Məşğələ 23 tamamlandı:** bütün məcburi code cells yuxarıdan aşağıya interaktiv giriş və gizli skip olmadan icra edilir.
